## Import Libraries and Data

The following code matches RAD/PACT addresses as of July 2025 to [Open Mark Orders](https://data.cityofnewyork.us/Housing-Development/Open-Market-Order-OMO-Charges/mdbu-nrqn/about_data) published by the NYC Department of Housing Preservation and Development.
<br><br>
According to HPD:
<br>"work orders are created to conduct emergency repair work when an owner fails to address a hazardous condition pursuant to the requirements of an HPD-issued violation ... or an emergency violation issued by another City Agency."

In [1]:
## import libraries
import pandas as pd
import numpy as np

In [2]:
## import data
omo_df = pd.read_csv('../input/Open_Market_Order_(OMO)_Charges_20251202.csv', dtype = {'OMOID':'object',
                                                                                       'BIN':'object',
                                                                                       'OMONumber':'object',
                                                                                       'BuildingID':'object',
                                                                                       'Boro ID':'object',
                                                                                       'BBL':'object',
                                                                                       'Block':'object',
                                                                                       'Lot':'object',
                                                                                       'HouseNumber':'object',
                                                                                       'Zip':'object',
                                                                                       'IsAEP':'object',
                                                                                       'IsCommercialDemolition':'object',
                                                                                       'FEMAEvent':'object',
                                                                                       'FEMAEvenID':'object'})

/var/folders/t4/yhv9cps51w7bfdt0w1lkv1zm0000gn/T/ipykernel_71212/2616853611.py:2: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  omo_df = pd.read_csv('../input/Open_Market_Order_(OMO)_Charges_20251202.csv', dtype = {'OMOID':'object',


In [3]:
omo_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 494233 entries, 0 to 494232
Data columns (total 32 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   OMOID                   494233 non-null  object 
 1   OMONumber               494233 non-null  object 
 2   BuildingID              494233 non-null  object 
 3   Boro ID                 494233 non-null  object 
 4   Boro                    494233 non-null  object 
 5   HouseNumber             494233 non-null  object 
 6   StreetName              494233 non-null  object 
 7   Apartment               371362 non-null  object 
 8   Zip                     494002 non-null  object 
 9   Block                   494233 non-null  object 
 10  Lot                     494233 non-null  object 
 11  LifeCycle               494233 non-null  object 
 12  WorkTypeGeneral         494233 non-null  object 
 13  OMOStatusReason         485854 non-null  object 
 14  OMOAwardAmount      

In [4]:
## read in the developments data
developments = pd.read_csv('../input/coded_files/addresses_updated_121525.csv',dtype={'bbl':'object',
                                                                                    'house_num':'object',
                                                                                    'bbl':'object',
                                                                                    'bin':'object'})

In [5]:
developments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1531 entries, 0 to 1530
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   bbl          1508 non-null   object 
 1   bldg         1431 non-null   float64
 2   m            1127 non-null   object 
 3   og_address   1530 non-null   object 
 4   zip code     1527 non-null   float64
 5   cd           1527 non-null   float64
 6   fc           1527 non-null   float64
 7   ss           1527 non-null   float64
 8   sa           1527 non-null   float64
 9   cc           1527 non-null   float64
 10  bin          1508 non-null   object 
 11  dev_name     1531 non-null   object 
 12  boro         1531 non-null   object 
 13  units        1531 non-null   object 
 14  transfer     1531 non-null   object 
 15  new_address  1530 non-null   object 
dtypes: float64(7), object(9)
memory usage: 191.5+ KB


## Cleaning

In [6]:
## rename address/boro columns so we can match later
developments = developments.rename(columns = {'new_address':'full_address',
                                              'borough':'boro'})

In [7]:
## make all columns in the OMO df all lowercase (for matching)
omo_df.columns = omo_df.columns.str.lower()

In [8]:
## stripping of whitespace
developments['bbl'] = developments['bbl'].str.strip()
developments['bin'] = developments['bin'].str.strip()
developments['full_address'] = developments['full_address'].str.strip()


In [9]:
## stripping of whitespace
omo_df['streetname'] = omo_df['streetname'].str.strip()
omo_df['housenumber'] = omo_df['housenumber'].str.strip()
omo_df['bbl'] = omo_df['bbl'].str.strip()
omo_df['bin'] = omo_df['bin'].str.strip()

In [10]:
## create a full street address in the OMO data by combining house number and street name
omo_df['full_address'] = omo_df['housenumber'] + ' ' + omo_df['streetname']

In [11]:
omo_df.boro.unique()

array(['Bronx', 'Queens', 'Brooklyn', 'Manhattan', 'Staten Island'],
      dtype=object)

In [12]:
developments.boro.unique()

array(['brooklyn', 'bronx', 'manhattan', 'queens', 'staten island'],
      dtype=object)

In [13]:
## create a dict of boroughs
omo_boro_changes = {'Bronx':'bronx',
                'Queens':'queens',
                'Manhattan':'manhattan',
                'Brooklyn':'brooklyn',
                'Staten Island':'staten island'}

In [14]:
## replace previous spellings with new spellings
omo_df['boro'] = omo_df['boro'].replace(omo_boro_changes)

## Merging

In [15]:
## combine on full address and borough, inner so we drop the developments with no data
combined_df = pd.merge(omo_df,
                       developments,
                       on = ['full_address','boro'],
                       how = 'left',
                       indicator=True)

In [16]:
## filter for only rows that are in both datasets
both_df = combined_df[combined_df['_merge']=='both'].reset_index()

In [17]:
both_df

,index,omoid,omonumber,buildingid,boro id,boro,housenumber,streetname,apartment,zip,...,cd,fc,ss,sa,cc,bin_y,dev_name,units,transfer,_merge
0,7,5371519,EO08654,809554,3,brooklyn,726,STANLEY AVENUE,NaN,11207,...,5.0,8.0,19.0,60.0,42.0,3324277,BOULEVARD,"1,441",12/28/2021,both
1,667,5298572,EN12278,809926,3,brooklyn,245,WORTMAN AVENUE,13D,11207,...,5.0,8.0,19.0,60.0,42.0,3324016,LINDEN,"1,586",12/28/2021,both
2,4243,5301673,EN13170,808855,3,brooklyn,320,MILLER AVENUE,NaN,11207,...,5.0,8.0,19.0,54.0,42.0,3327009,FIORENTINO PLAZA,160,12/28/2021,both
3,4277,5353121,EO02877,43031,1,manhattan,463,WEST 164 STREET,NaN,10032,...,12.0,13.0,31.0,72.0,10.0,1062641,WASHINGTON HEIGHTS REHAB PHASE III (FORT WASHI...,88,11/30/2020,both
4,4288,5353128,EO02882,43298,1,manhattan,509,WEST 176 STREET,NaN,10033,...,12.0,13.0,31.0,72.0,10.0,1063207,WASHINGTON HEIGHTS REHAB (GROUPS 1&2),216,11/30/2020,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,493373,5529876,EP28137,380678,3,brooklyn,104,TAPSCOTT STREET,NaN,11212,...,16.0,9.0,25.0,55.0,41.0,3081191,104-14 TAPSCOTT STREET,30,11/28/2023,both
409,493663,5530995,EP28738,363151,3,brooklyn,1142,LENOX ROAD,NaN,11212,...,17.0,9.0,19.0,58.0,41.0,3101906,LENOX ROAD-ROCKAWAY PARKWAY,74,11/28/2023,both
410,493665,5531326,EP28901,806340,2,bronx,400,BROOK AVENUE,NaN,10454,...,1.0,14.0,29.0,84.0,8.0,2090994,BETANCES I,309,11/16/2018,both
411,493976,5531846,EP29155,41707,1,manhattan,145,WEST 143 STREET,4B,10030,...,10.0,13.0,30.0,71.0,9.0,1060133,SAMUEL (CITY),664,9/26/2024,both


In [18]:
## write to a csv
both_df.to_csv('../output/omo_data_12152025.csv')

In [19]:
both_df.dev_name.value_counts().head(10)

dev_name
BOULEVARD                                   48
LINDEN                                      36
EDENWALD                                    33
OCEAN BAY APARTMENTS (BAYSIDE)              20
SAMUEL (CITY)                               18
STERLING PLACE REHABS (STERLING-BUFFALO)    17
PARK ROCK REHAB                             15
MANHATTANVILLE                              12
FIORENTINO PLAZA                            11
HARLEM RIVER                                11
Name: count, dtype: int64

In [20]:
## number of unique developments with at least one OMO
both_df.dev_name.nunique()

60

In [21]:
## number of unique OMOs
both_df.omoid.nunique()

410

## OMOs between January 1, 2021, and September 25, 2025.

In [22]:
## change dtype for the dates in which the OMOs were created
both_df['omocreatedate'] = both_df['omocreatedate'].astype('datetime64[ns]')

In [23]:
omos_21_25 = both_df[(both_df['omocreatedate'] >= '01-01-2021') & (both_df['omocreatedate'] <= '09-25-2025')]

In [24]:
omos_21_25.shape

(372, 49)

In [25]:
omos_21_25.omoid.nunique()

369

In [26]:
omos_21_25.to_csv('../output/testing_omos_121525.csv')

In [27]:
omos_21_25.dev_name.nunique()

59

In [28]:
omos_21_25.omocreatedate.dt.year.value_counts().reset_index()

,omocreatedate,count
0,2024,118
1,2025,99
2,2023,85
3,2022,54
4,2021,16


## Detour: Looking at Boulevard

In [29]:
boulevard_df = both_df[both_df['dev_name'] == 'BOULEVARD']

In [30]:
boulevard_df.omocreatedate.dt.year.value_counts().reset_index()

,omocreatedate,count
0,2023,19
1,2024,16
2,2025,7
3,2022,6


## Looking at costs

In [31]:
omos_21_25['omoawardamount'] = pd.to_numeric(omos_21_25['omoawardamount'], errors='coerce')

/var/folders/t4/yhv9cps51w7bfdt0w1lkv1zm0000gn/T/ipykernel_71212/946655148.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omos_21_25['omoawardamount'] = pd.to_numeric(omos_21_25['omoawardamount'], errors='coerce')


In [32]:
median_cost = omos_21_25.groupby('dev_name')['omoawardamount'].median()
median_cost.head()

dev_name
104-14 TAPSCOTT STREET             490.00
344 EAST 28TH STREET               333.87
ARMSTRONG I                        250.25
ARMSTRONG II                       780.00
BAILEY AVENUE-WEST 193RD STREET       NaN
Name: omoawardamount, dtype: float64

In [33]:
total_cost = omos_21_25.groupby('dev_name')['omoawardamount'].sum().reset_index(name = 'total_cost')
total_cost.head()

,dev_name,total_cost
0,104-14 TAPSCOTT STREET,490.00
1,344 EAST 28TH STREET,667.74
2,ARMSTRONG I,500.50
3,ARMSTRONG II,780.00
4,BAILEY AVENUE-WEST 193RD STREET,0.00
